## CS336 playground

In [1]:
import time
import torch
import os
import cs336_basics.ron_adamw_optimizer as ron_adamw_optimizer
import cs336_basics.ron_bpe_tokenizer as ron_bpe_tokenizer
import cs336_basics.ron_causal_multihead_self_attention_with_rope as ron_causal_multihead_self_attention_with_rope
import cs336_basics.ron_cross_entropy as ron_cross_entropy
import cs336_basics.ron_data_loader as ron_data_loader
import cs336_basics.ron_embedding as ron_embedding
import cs336_basics.ron_linear as ron_linear
import cs336_basics.ron_multihead_self_attention as ron_multihead_self_attention
import cs336_basics.ron_rmsnorm as ron_rmsnorm
import cs336_basics.ron_rope as ron_rope
import cs336_basics.ron_scaled_dot_product_attention as ron_scaled_dot_product_attention
import cs336_basics.ron_softmax as ron_softmax
import cs336_basics.ron_swiglu as ron_swiglu
import cs336_basics.ron_train_bpe as ron_train_bpe
import cs336_basics.ron_transformer_lm as ron_transformer_lm

In [2]:
# from the 7.2 secion

vocab_size = 10000
context_length = 256
d_model = 512
d_ff = 1344
rope_theta = 10000
num_layers = 4
num_heads = 16
batch_size = 16 # fits in 6GB Nvidia 2060
device = (
    (torch.cuda.is_available() and 'cuda') or
    (torch.backends.mps.is_available() and 'mps') or
    "cpu"
)


!wget -q -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O /tmp/tiny_shakespeare.txt
!wget -q -nc https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt -O ../data/TinyStoriesV2-GPT4-train.txt
!wget -q -nc https://dgoldberg.sdsu.edu/515/harrypotter.txt -O /tmp/harrypotter.txt

datasets = {
   'tinystories'           : '../data/TinyStoriesV2-GPT4-train.txt',
   'shakespeare'           : '/tmp/tiny_shakespeare.txt',
   'book'                  : '/tmp/harrypotter.txt',
   'tinystories_valudation': '../data/TinyStoriesV2-GPT4-train.txt',
}    
if shakespeare:=True:
    vocab_source = datasets['shakespeare']
    vocab_cache = f"/tmp/bpe_shakespeare_{vocab_size}.saved"
    training_txt = datasets['shakespeare']
    training_uint16 = "/tmp/tiny_shakespeare.uint16"

if tinystories:=False:
    vocab_source = datasets['tinystories_validation']
    vocab_cache = f"/tmp/bpe_tinystories_{vocab_size}.saved"
    training_txt = '../data/TinyStoriesV2-GPT4-train.txt'
    training_uint16 = "/tmp/tiny_stories_training.uint16"

if fine_tune:=False:
    vocab_source = '../data/TinyStoriesV2-GPT4-valid.txt'
    vocab_cache = f"/tmp/bpe_tinystories_{vocab_size}.saved"
    training_txt = '/tmp/harrypotter.txt'
    training_uint16 = f"/tmp/harrypotter_tinystories_bpe.uint16"



## BPE training

In [3]:
"""
    vocab size of 1000 on TinyStoriesV2-GPT4-valid.txt  
    isn't horrible for CPU-based interactive notebook use.
    It takes 16 seconds to train and makes words like " Sally" and " helped" 
    into single tokens. Maybe roughly the vocab of a preschooler :)

    vocab size of 10000 on TinyStoriesV2-GPT4-valid.txt takes like 
    three minutes. and makes words like " sinking" and " ribbit".

    Shakespeare's about 3 minutes as well. 
"""
if not os.path.exists(vocab_cache):
    vocab,merges = ron_train_bpe.train_bpe(vocab_source,vocab_size,[])
    obj = {"vocab": vocab,"merges": merges}
    torch.save(obj, vocab_cache)
else:
    obj = torch.load(vocab_cache)
    vocab,merges = obj['vocab'],obj['merges']
print(merges[-10:])

[(b'read', b'ing'), (b're', b'qu'), (b're', b'cy'), (b're', b'ck'), (b're', b'b'), (b're', b'aming'), (b're', b'am'), (b'ray', b'be'), (b'rant', b's'), (b'ran', b'g')]


## BPE class

In [4]:
tokenizer = ron_bpe_tokenizer.RonBPETokenizer(vocab,merges)
tokens    = list(tokenizer.encode_iterable(["Hello world"," ","Good bye"]))
print(tokens)
print(tokenizer.decode(tokens))

[72, 408, 111, 864, 32, 1227, 415, 101]
Hello world Good bye


### Make a nice training dataset with that tokenizer

In [5]:
import numpy as np
import tqdm
def tokenize_file_to_numpy(in_path, out_path, tokenizer, dtype=np.uint16):
    tokens = []
    with open(out_path, "ab") as fout:
        with open(in_path, "r", encoding="utf-8") as fin:
            for line in tqdm.tqdm(fin):
                toks = tokenizer.encode(line + "\n")
                toks_np = np.asarray(toks, dtype=dtype)
                fout.write(toks_np.tobytes())

# about 7 seconds with a 1000 vocab
# about 30 seconds with a 10000 vocab
# longer on more diverse documents (with more distinct pretokenized strings)

if not os.path.exists(training_uint16):
    tokenize_file_to_numpy(training_txt,training_uint16,tokenizer)

token_ds = np.memmap(training_uint16, dtype=np.uint16, mode="r")
token_ds

memmap([ 672, 1196,   58, ...,   46,   10,   10],
       shape=(352087,), dtype=uint16)

## My Transformer model


In [6]:
import cs336_basics.ron_transformer_lm as ron_transformer_lm

tlm = ron_transformer_lm.TransformerLM(
    d_model=d_model,
    num_heads=num_heads,
    d_ff = d_ff,
    max_seq_len=context_length,
    theta=rope_theta,
    vocab_size=vocab_size,
    context_length=context_length,
    num_layers=num_layers
    )
tlm.to(device)
tlm.forward([tokenizer.encode("Hello world")])

tensor([[[-0.3076, -0.0209,  0.5002,  ...,  0.3429, -0.1164, -0.2557],
         [-0.1129,  0.4348,  0.0821,  ...,  0.3225,  0.0355, -0.1493],
         [-0.2026,  0.2474,  0.2501,  ...,  0.8173,  0.0591,  0.1453],
         [-0.2878,  0.3563, -0.0421,  ...,  0.1360, -0.3383, -0.1135]]],
       device='cuda:0', grad_fn=<ViewBackward0>)

### Predict words

* Expected to be quite random here - no training occurred

In [7]:
def predict_words(
        tlm: ron_transformer_lm.TransformerLM,
        tokenizer: ron_bpe_tokenizer.RonBPETokenizer,
        num_tokens_to_generate = 50,
        initial_text: str = "One day",
):
    generated_tokens = tokenizer.encode(initial_text)
    for _ in range(num_tokens_to_generate):
        seq_len = len(generated_tokens)
        tokens_tensor = torch.tensor([generated_tokens])
        with torch.no_grad():
            predictions = tlm.forward(tokens_tensor)  # (1, seq_len, vocab_size)
        last_logits = predictions[0, -1]
        #next_token_id = last_logits.argmax().item()
        probs = ron_softmax.softmax(last_logits,-1)
        next_token_id = torch.multinomial(probs, num_samples=1).item()
        generated_tokens.append(next_token_id)

    return tokenizer.decode(generated_tokens)

predict_words(tlm,tokenizer,40)

'One day mastrangeimes appoint needVALERIA eatenothingings pleasant numbereyWorseuel poxSit swear print hon annoywinds coming divisionph Leave loves Apollocing defVIR sped rule songRUTLANDifteenfendbanishedhireps den'

## My AdamW Optimizer

In [8]:
my_adamw = ron_adamw_optimizer.AdamW(tlm.parameters())

## My Data Loader

In [9]:
def get_training_batch():
    x,y = ron_data_loader.get_batch(token_ds,batch_size,context_length,device)
    if device == 'mps':
        # TypeError: Trying to convert UInt16 to the MPS backend but it does not have support for that dtype.
        x = x.to('cpu').to(dtype=torch.long).to(device)
        y = y.to('cpu').to(dtype=torch.long).to(device)
    else:
        x = x.to(dtype=torch.long)
        y = y.to(dtype=torch.long)
    return x,y


### Compile the model

Can take 14 seconds for the first forward pass...

Quoting a chatbot: 

> PyTorch (via TorchDynamo + TorchInductor) traces your model the first time you call it, captures the computation graph, and then compiles optimized kernels for your GPU.
>
> What happens:
>
> * First call (slow, e.g. 14s)
>   * TorchDynamo intercepts your Python code.
>   * TorchInductor generates optimized code (usually CUDA kernels).
>   * Compilation is JIT (just-in-time), so it can take several seconds.
> * Subsequent calls (fast, e.g. 0.2s)
>   * The compiled kernels are cached.
>   * PyTorch skips Python dispatch and just runs the optimized kernels.
>
> That’s why you see a huge difference between the first iteration and the rest.

This warning is expected on my cheap old GPU

```
W0901 17:10:39.072000 3071772 torch/_inductor/utils.py:1137] [0/0] Not enough SMs to use max_autotune_gemm mode
```




In [10]:
# warmup to not skew training iter time
tlm.compile()
tokens = torch.randint(0, vocab_size, (16, 256), device="cuda")
tlm(tokens)
pass

## Training Loop

In [11]:
import mlflow
import IPython.display as ipd
import html
import regex as re
def train(model, optimizer, get_batch, device="cuda", max_training_time = 60 * 5, num_iters=1000*1000):
    model.to(device)
    model.train()

    with mlflow.start_run():
        mlflow.log_params({
            "device": device,
            "num_iters": num_iters,
            "optimizer": type(optimizer).__name__,
            "model": type(model).__name__
        })
        t0 = time.time()
        next_log_interval = 1
        for it in range(num_iters):
            x, y = get_batch()
            logits = model(x)            # shape: (batch, seq, vocab)
            loss = ron_cross_entropy.cross_entropy(
                logits.view(-1, logits.size(-1)),
                y.view(-1)
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if it >= next_log_interval:
                next_log_interval *= 2
                mlflow.log_metric("loss", loss.item(), step=it)
                mlflow.log_metric("elapsed_time", time.time() - t0, step=it)
                output = predict_words(tlm,tokenizer,100)
                ipd.display(ipd.HTML(f'''
                    <style> td {{text-align: left;}} </style>
                    <table>
                        <td width="20%">iter {it}<br>loss {loss.item():.4f}<br>{time.time() - t0:.3f} secs</td>
                        <td>{re.sub("\n+","<br>",html.escape(output))}</td></tr>
                    </table>
                '''))
                if time.time() - t0 > max_training_time:
                    break

In [12]:
train(tlm,my_adamw,get_training_batch,max_training_time=60*34)

KeyboardInterrupt: 

## Or an example trained on tinystories

In [ ]:

train(tlm,my_adamw,get_training_batch,max_training_time=60*30)